<a href="https://colab.research.google.com/github/sidharth2733mba25fin-ops/COLLAB-FILES/blob/main/Lab_5_6_Ethical_Risk_Finance_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5–6: Ethical Risk Analysis in AI-Based Finance Systems

## Topic
Ethical Risk Detection using a Synthetic Finance Dataset

## Learning Objectives
By the end of this lab, students will be able to:

- Understand ethical risks in AI-based finance decisions.
- Create and analyze a synthetic finance dataset.
- Identify privacy, fairness, transparency, and automation risks.
- Build an ethical risk score for financial AI use cases.
- Classify cases into Low, Medium, and High Ethical Risk categories.
- Interpret risk patterns using visualizations.

## Business Scenario
A fintech company is using AI to evaluate digital loan applications. The system uses financial data, digital behavior, customer segment, location type, and consent information. Management wants to know whether the system creates ethical risks before full deployment.


In [ ]:
# Cell 1: Import required libraries
# pandas is used for creating and analyzing tabular data.
# numpy is used for numerical operations and random data generation.
# matplotlib and seaborn are used for visualization.
# sklearn is used for simple preprocessing and model-based analysis.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# This setting makes charts appear inside the notebook.
%matplotlib inline

# This makes the random dataset reproducible.
# Reproducible means students will get the same results each time they run the notebook.
np.random.seed(42)


In [ ]:
# Cell 2: Create a new synthetic finance dataset
# This dataset is NOT real customer data.
# It is created only for teaching ethical risk analysis.
# Each row represents one digital loan application reviewed by an AI system.

num_records = 1200

customer_id = np.arange(1, num_records + 1)

# Customer segment shows how formally connected the customer is with the financial system.
customer_segment = np.random.choice(
    ['Salaried Formal', 'Self-Employed', 'Gig Worker', 'Small Trader', 'Student/New-to-Credit'],
    size=num_records,
    p=[0.35, 0.20, 0.15, 0.20, 0.10]
)

# Location type helps us analyze whether certain geographies face higher ethical risk.
location_type = np.random.choice(
    ['Metro', 'Tier-2 City', 'Tier-3/Rural'],
    size=num_records,
    p=[0.45, 0.35, 0.20]
)

# Language preference may affect form completion and chatbot understanding.
language_preference = np.random.choice(
    ['English', 'Hindi', 'Regional Language'],
    size=num_records,
    p=[0.50, 0.30, 0.20]
)

# Device category is used by some digital lenders as an alternative behavioral signal.
# This can create ethical risk if device cost becomes a proxy for income or class.
device_category = np.random.choice(
    ['Premium Smartphone', 'Mid-range Smartphone', 'Budget Smartphone'],
    size=num_records,
    p=[0.20, 0.50, 0.30]
)

# Consent status shows whether the customer clearly consented to data usage.
# Lack of clear consent increases privacy and governance risk.
consent_status = np.random.choice(
    ['Clear Consent', 'Partial Consent', 'No Clear Consent'],
    size=num_records,
    p=[0.65, 0.25, 0.10]
)

# Human review indicates whether a human officer reviewed the AI decision.
# High-risk decisions without human review may create accountability concerns.
human_review = np.random.choice(
    ['Reviewed', 'Not Reviewed'],
    size=num_records,
    p=[0.40, 0.60]
)

# Monthly income is generated differently for each segment to make data realistic.
monthly_income = []
for seg in customer_segment:
    if seg == 'Salaried Formal':
        monthly_income.append(np.random.normal(75000, 18000))
    elif seg == 'Self-Employed':
        monthly_income.append(np.random.normal(90000, 35000))
    elif seg == 'Gig Worker':
        monthly_income.append(np.random.normal(45000, 15000))
    elif seg == 'Small Trader':
        monthly_income.append(np.random.normal(65000, 30000))
    else:
        monthly_income.append(np.random.normal(30000, 10000))
monthly_income = np.clip(monthly_income, 12000, 250000).round(0)

# Credit history months measures how long the customer has formal credit history.
credit_history_months = []
for seg in customer_segment:
    if seg == 'Student/New-to-Credit':
        credit_history_months.append(np.random.randint(0, 10))
    elif seg == 'Gig Worker':
        credit_history_months.append(np.random.randint(3, 36))
    else:
        credit_history_months.append(np.random.randint(6, 120))

# Debt-to-income ratio estimates how much of income is already committed to debt.
debt_to_income_ratio = np.random.beta(2, 5, num_records)
debt_to_income_ratio = np.round(debt_to_income_ratio, 2)

# Digital activity score represents app usage, form completion quality, and digital engagement.
# Important ethical note: this may not always represent true creditworthiness.
digital_activity_score = []
for device in device_category:
    if device == 'Premium Smartphone':
        digital_activity_score.append(np.random.normal(78, 10))
    elif device == 'Mid-range Smartphone':
        digital_activity_score.append(np.random.normal(62, 14))
    else:
        digital_activity_score.append(np.random.normal(48, 18))
digital_activity_score = np.clip(digital_activity_score, 0, 100).round(0)

# Data sensitivity score represents how sensitive the data used by the AI system is.
# Higher score means more personal or sensitive data usage.
data_sensitivity_score = np.random.randint(20, 96, size=num_records)

# Explainability score shows how clearly the AI system can explain its decision.
# Lower score means higher black-box risk.
explainability_score = np.random.randint(25, 96, size=num_records)

# Create the DataFrame.
df = pd.DataFrame({
    'customer_id': customer_id,
    'customer_segment': customer_segment,
    'location_type': location_type,
    'language_preference': language_preference,
    'device_category': device_category,
    'consent_status': consent_status,
    'human_review': human_review,
    'monthly_income': monthly_income,
    'credit_history_months': credit_history_months,
    'debt_to_income_ratio': debt_to_income_ratio,
    'digital_activity_score': digital_activity_score,
    'data_sensitivity_score': data_sensitivity_score,
    'explainability_score': explainability_score
})

# Display the first five records.
df.head()


,customer_id,customer_segment,location_type,language_preference,device_category,consent_status,human_review,monthly_income,credit_history_months,debt_to_income_ratio,digital_activity_score,data_sensitivity_score,explainability_score
0,1,Self-Employed,Tier-2 City,English,Premium Smartphone,Clear Consent,Not Reviewed,56709.0,77,0.43,82.0,41,57
1,2,Student/New-to-Credit,Metro,English,Mid-range Smartphone,Clear Consent,Reviewed,26582.0,8,0.27,55.0,44,49
2,3,Small Trader,Metro,English,Premium Smartphone,No Clear Consent,Not Reviewed,34792.0,76,0.17,100.0,78,38
3,4,Gig Worker,Metro,English,Mid-range Smartphone,Clear Consent,Not Reviewed,45717.0,16,0.34,56.0,76,62
4,5,Salaried Formal,Tier-2 City,Hindi,Mid-range Smartphone,Partial Consent,Reviewed,64844.0,33,0.42,75.0,73,62


In [ ]:
# Cell 3: Create an AI loan decision score
# This score simulates how an AI lending model may decide approval.
# Higher income, longer credit history, and higher digital activity increase the score.
# Higher debt-to-income ratio decreases the score.
# Note: This is intentionally simplified for classroom learning.

df['ai_decision_score'] = (
    0.00025 * df['monthly_income'] +
    0.25 * df['credit_history_months'] -
    35 * df['debt_to_income_ratio'] +
    0.30 * df['digital_activity_score']
)

# Convert score into loan decision.
# If score is above 45, application is approved; otherwise rejected.
df['loan_decision'] = np.where(df['ai_decision_score'] >= 45, 'Approved', 'Rejected')

# Show selected columns to understand the decision output.
df[['customer_id', 'customer_segment', 'monthly_income', 'credit_history_months',
    'debt_to_income_ratio', 'digital_activity_score', 'ai_decision_score', 'loan_decision']].head(10)


,customer_id,customer_segment,monthly_income,credit_history_months,debt_to_income_ratio,digital_activity_score,ai_decision_score,loan_decision
0,1,Self-Employed,56709.0,77,0.43,82.0,42.97725,Rejected
1,2,Student/New-to-Credit,26582.0,8,0.27,55.0,15.69550,Rejected
2,3,Small Trader,34792.0,76,0.17,100.0,51.74800,Approved
3,4,Gig Worker,45717.0,16,0.34,56.0,20.32925,Rejected
4,5,Salaried Formal,64844.0,33,0.42,75.0,32.26100,Rejected
5,6,Salaried Formal,48107.0,7,0.19,74.0,29.32675,Rejected
6,7,Salaried Formal,97593.0,48,0.66,53.0,29.19825,Rejected
7,8,Small Trader,50070.0,13,0.09,73.0,34.51750,Rejected
8,9,Gig Worker,25978.0,15,0.04,38.0,20.24450,Rejected
9,10,Small Trader,59952.0,90,0.31,36.0,37.43800,Rejected


In [ ]:
# Cell 4: Create ethical risk indicators
# We now create separate risk flags related to privacy, explainability, fairness, and accountability.
# These are not financial risk indicators; they are ethical risk indicators.

# Privacy risk is high when sensitive data is used and consent is not clear.
df['privacy_risk_flag'] = np.where(
    (df['data_sensitivity_score'] > 70) & (df['consent_status'] != 'Clear Consent'),
    1,
    0
)

# Explainability risk is high when explanation score is low.
df['explainability_risk_flag'] = np.where(df['explainability_score'] < 50, 1, 0)

# Fairness risk is high when vulnerable or underrepresented segments are rejected.
# This is a simplified educational rule.
vulnerable_segments = ['Gig Worker', 'Small Trader', 'Student/New-to-Credit']
df['fairness_risk_flag'] = np.where(
    (df['customer_segment'].isin(vulnerable_segments)) & (df['loan_decision'] == 'Rejected'),
    1,
    0
)

# Accountability risk is high when a rejected case has no human review.
df['accountability_risk_flag'] = np.where(
    (df['loan_decision'] == 'Rejected') & (df['human_review'] == 'Not Reviewed'),
    1,
    0
)

# Display risk flags.
df[['customer_id', 'loan_decision', 'privacy_risk_flag', 'explainability_risk_flag',
    'fairness_risk_flag', 'accountability_risk_flag']].head(10)


,customer_id,loan_decision,privacy_risk_flag,explainability_risk_flag,fairness_risk_flag,accountability_risk_flag
0,1,Rejected,0,0,0,1
1,2,Rejected,0,1,1,0
2,3,Approved,1,1,0,0
3,4,Rejected,0,0,1,1
4,5,Rejected,1,0,0,0
5,6,Rejected,0,1,0,1
6,7,Rejected,0,0,0,1
7,8,Rejected,1,1,1,0
8,9,Rejected,0,0,1,1
9,10,Rejected,0,1,1,0


In [ ]:
# Cell 5: Build an overall ethical risk score
# Each risk area receives a weight.
# Higher total score means higher ethical risk.
# Weights can be changed during class discussion.

df['ethical_risk_score'] = (
    30 * df['privacy_risk_flag'] +
    25 * df['explainability_risk_flag'] +
    25 * df['fairness_risk_flag'] +
    20 * df['accountability_risk_flag']
)

# Convert numerical score into risk categories.
def risk_category(score):
    if score >= 60:
        return 'High Ethical Risk'
    elif score >= 30:
        return 'Medium Ethical Risk'
    else:
        return 'Low Ethical Risk'

df['ethical_risk_category'] = df['ethical_risk_score'].apply(risk_category)

# Display results.
df[['customer_id', 'loan_decision', 'ethical_risk_score', 'ethical_risk_category']].head(10)


,customer_id,loan_decision,ethical_risk_score,ethical_risk_category
0,1,Rejected,20,Low Ethical Risk
1,2,Rejected,50,Medium Ethical Risk
2,3,Approved,55,Medium Ethical Risk
3,4,Rejected,45,Medium Ethical Risk
4,5,Rejected,30,Medium Ethical Risk
5,6,Rejected,45,Medium Ethical Risk
6,7,Rejected,20,Low Ethical Risk
7,8,Rejected,80,High Ethical Risk
8,9,Rejected,45,Medium Ethical Risk
9,10,Rejected,50,Medium Ethical Risk


In [ ]:
# Cell 6: Check dataset shape and basic information
# This helps students understand how many rows and columns are present.

print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nDataset information:')
df.info()

Dataset shape: (1200, 21)

Column names:
['customer_id', 'customer_segment', 'location_type', 'language_preference', 'device_category', 'consent_status', 'human_review', 'monthly_income', 'credit_history_months', 'debt_to_income_ratio', 'digital_activity_score', 'data_sensitivity_score', 'explainability_score', 'ai_decision_score', 'loan_decision', 'privacy_risk_flag', 'explainability_risk_flag', 'fairness_risk_flag', 'accountability_risk_flag', 'ethical_risk_score', 'ethical_risk_category']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               1200 non-null   int64  
 1   customer_segment          1200 non-null   object 
 2   location_type             1200 non-null   object 
 3   language_preference       1200 non-null   object 
 4   device_category           1200 non-null

In [ ]:
# Cell 7: Summary statistics for numerical columns
# This helps students understand the distribution of income, credit history, scores, and risk values.

df.describe()


In [ ]:
# Cell 8: Ethical risk category distribution
# This shows how many cases fall into Low, Medium, and High Ethical Risk groups.

risk_counts = df['ethical_risk_category'].value_counts()
print(risk_counts)

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='ethical_risk_category', order=['Low Ethical Risk', 'Medium Ethical Risk', 'High Ethical Risk'])
plt.title('Distribution of Ethical Risk Categories')
plt.xlabel('Ethical Risk Category')
plt.ylabel('Number of Applications')
plt.xticks(rotation=15)
plt.show()


In [ ]:
# Cell 9: Loan decision distribution by customer segment
# This helps identify whether some segments face higher rejection.
# In ethical AI analysis, group-level comparison is very important.

segment_decision = pd.crosstab(df['customer_segment'], df['loan_decision'], normalize='index') * 100
segment_decision = segment_decision.round(2)
segment_decision


In [ ]:
# Cell 10: Visualize approval and rejection rates by customer segment
# This chart helps students visually compare outcomes across groups.

segment_decision.plot(kind='bar', figsize=(10, 6))
plt.title('Loan Decision Percentage by Customer Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Percentage')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Loan Decision')
plt.show()


In [ ]:
# Cell 11: Ethical risk by customer segment
# This shows whether some customer groups are carrying more ethical risk.

segment_risk = pd.crosstab(df['customer_segment'], df['ethical_risk_category'], normalize='index') * 100
segment_risk = segment_risk.round(2)
segment_risk


In [ ]:
# Cell 12: Visualize ethical risk by customer segment
# A high percentage of ethical risk in one segment may indicate bias, poor explainability, or inadequate review.

segment_risk.plot(kind='bar', stacked=True, figsize=(11, 6))
plt.title('Ethical Risk Category by Customer Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Percentage')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Ethical Risk Category')
plt.show()


In [ ]:
# Cell 13: Ethical risk by location type
# This helps identify whether geography is connected with higher ethical risk.
# In finance, location can become a proxy variable if not handled carefully.

location_risk = pd.crosstab(df['location_type'], df['ethical_risk_category'], normalize='index') * 100
location_risk = location_risk.round(2)
location_risk


In [ ]:
# Cell 14: Privacy risk analysis
# This identifies how many customers have privacy risk due to sensitive data usage without clear consent.

privacy_summary = pd.crosstab(df['consent_status'], df['privacy_risk_flag'], normalize='index') * 100
privacy_summary = privacy_summary.round(2)
privacy_summary.columns = ['No Privacy Risk %', 'Privacy Risk %']
privacy_summary


In [ ]:
# Cell 15: Explainability risk analysis
# Low explainability can create black-box risk.
# This is especially serious in rejected loan decisions because customers deserve clear reasons.

explainability_by_decision = df.groupby('loan_decision')['explainability_score'].mean().round(2)
print(explainability_by_decision)

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='loan_decision', y='explainability_score')
plt.title('Explainability Score by Loan Decision')
plt.xlabel('Loan Decision')
plt.ylabel('Explainability Score')
plt.show()


In [ ]:
# Cell 16: Accountability risk analysis
# This checks how many rejected applications did not receive human review.
# In high-stakes finance decisions, human review is important for accountability.

rejected_cases = df[df['loan_decision'] == 'Rejected']
accountability_table = rejected_cases['human_review'].value_counts(normalize=True) * 100
accountability_table = accountability_table.round(2)
print(accountability_table)

plt.figure(figsize=(7, 5))
sns.countplot(data=rejected_cases, x='human_review')
plt.title('Human Review Status for Rejected Applications')
plt.xlabel('Human Review Status')
plt.ylabel('Number of Rejected Applications')
plt.show()


In [ ]:
# Cell 17: Identify high ethical risk cases
# These are the cases that management should investigate first.
# We sort by ethical risk score in descending order.

high_risk_cases = df[df['ethical_risk_category'] == 'High Ethical Risk'].copy()
high_risk_cases = high_risk_cases.sort_values(by='ethical_risk_score', ascending=False)

high_risk_cases[['customer_id', 'customer_segment', 'location_type', 'consent_status',
                 'human_review', 'loan_decision', 'ethical_risk_score',
                 'privacy_risk_flag', 'explainability_risk_flag',
                 'fairness_risk_flag', 'accountability_risk_flag']].head(20)


In [ ]:
# Cell 18: Create a simple machine learning model to predict high ethical risk
# This is not required for governance, but useful for learning.
# We create a target variable: 1 = High Ethical Risk, 0 = Not High Ethical Risk.

df['high_ethical_risk_target'] = np.where(df['ethical_risk_category'] == 'High Ethical Risk', 1, 0)

# Select features for prediction.
features = [
    'customer_segment', 'location_type', 'language_preference', 'device_category',
    'consent_status', 'human_review', 'monthly_income', 'credit_history_months',
    'debt_to_income_ratio', 'digital_activity_score', 'data_sensitivity_score',
    'explainability_score'
]

X = df[features]
y = df['high_ethical_risk_target']

# Convert categorical variables into numeric dummy variables.
X_encoded = pd.get_dummies(X, drop_first=True)

# Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.25, random_state=42, stratify=y
)

# Train a Random Forest model.
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on test data.
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('
Classification Report:')
print(classification_report(y_test, y_pred))


In [ ]:
# Cell 19: Feature importance for high ethical risk prediction
# Feature importance helps identify which variables are strongly associated with high ethical risk.
# This is useful for management interpretation and governance review.

feature_importance = pd.DataFrame({
    'feature': X_encoded.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

feature_importance.head(15)


In [ ]:
# Cell 20: Visualize top feature importances
# This chart shows the top variables influencing high ethical risk prediction.

top_features = feature_importance.head(12)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_features, x='importance', y='feature')
plt.title('Top Features Associated with High Ethical Risk')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()


In [ ]:
# Cell 21: Create management recommendations based on risk flags
# This cell translates technical findings into governance recommendations.
# MBA Finance students should learn how to convert analytics into boardroom actions.

def generate_recommendation(row):
    recommendations = []

    if row['privacy_risk_flag'] == 1:
        recommendations.append('Review consent and reduce sensitive data usage')
    if row['explainability_risk_flag'] == 1:
        recommendations.append('Improve reason codes and model explainability')
    if row['fairness_risk_flag'] == 1:
        recommendations.append('Conduct fairness audit for this customer segment')
    if row['accountability_risk_flag'] == 1:
        recommendations.append('Send case for human review before final rejection')

    if len(recommendations) == 0:
        return 'No immediate ethical risk action required'
    else:
        return '; '.join(recommendations)

df['management_recommendation'] = df.apply(generate_recommendation, axis=1)

df[['customer_id', 'ethical_risk_category', 'ethical_risk_score', 'management_recommendation']].head(15)


In [ ]:
# Cell 22: Export the final ethical risk dataset
# This file can be downloaded from Colab and submitted as lab output.

output_file = 'lab_5_6_ethical_risk_dataset.csv'
df.to_csv(output_file, index=False)

print('Dataset exported successfully as:', output_file)


# Lab Reflection Questions

Students should answer these after completing the notebook:

1. Which customer segment shows the highest ethical risk? Why might this be happening?
2. Is high accuracy enough for an AI system in finance? Why or why not?
3. How can unclear consent create privacy risk?
4. Why is human review important for rejected applications?
5. How can device category or location type become proxy variables?
6. What governance actions should management take before deploying this AI system?

# Final Learning Summary

- Ethical risk is different from credit risk.
- AI systems may be technically accurate but ethically weak.
- Privacy, explainability, fairness, and accountability must be monitored together.
- Finance leaders must convert technical findings into governance actions.
